<a href="https://colab.research.google.com/github/iamanzo1990/Movilidad-urbana-y-productividad-econ-mica-en-ciudades-de-LATAM/blob/main/S5_ladb_mobility_economy_project_student_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

-----

# Movilidad urbana y productividad económica en ciudades de LATAM

----

## Introducción

El objetivo de este proyecto es **evaluar cómo la movilidad urbana se relaciona con la productividad económica en las principales ciudades latinoamericanas**.
Para ello tomamos datos reales de TomTom Traffic Index y OECD Cities, que se deben limpiar, combinar y analizar para identificar en qué ciudades conviene invertir en infraestructura de transporte.

###Cargar y explorar

Previo a limpiar o combinar los datos, nos **familiaricemos con la estructura de ambos datasets**.
Validaremos que los archivos se carguen correctamente, conoceremos sus columnas y tipos de datos, y detectaremos posibles inconsistencias.

### 1.1 Carga de datos y vista rápida

**Objetivo:**
Importar las librerías necesarias, cargar los archivos CSV en DataFrames y realizar una revisión preliminar para entender su contenido.

**Pasos:**
- Importar las librerías `pandas`, `numpy`, `seaborn` y `matplotlib.pyplot`.
- Cargar los archivos usando `pd.read_csv()`:
  - `'/datasets/tomtom_traffic.csv'`
  - `/datasets/oecd_city_economy.csv` `.
- Guardar los DataFrames en las variables `traffic` y `eco`.
- Mostrar las primeras 5 filas de cada DataFrame.


In [9]:
# importar librerías

from google.colab import drive
drive.mount('/content/drive')


import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/TripleTen Projects/Career Project 1/ladb_mobility_economy_2024_clean (1).csv')

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# cargar archivos


traffic = pd.read_csv('/datasets/tomtom_traffic.csv')
eco = pd.read_csv('/datasets/oecd_city_economy.csv')



FileNotFoundError: [Errno 2] No such file or directory: '/datasets/tomtom_traffic.csv'

In [ ]:
# mostrar las primeras 5 filas de traffic
traffic.head()

In [ ]:
# mostrar las primeras 5 filas de eco
eco.head()


---

## Explorar, limpiar y preparar los datos

Antes de combinar los datasets, inspeccionamos su estructura, tipos de datos, columnas y valores faltantes.
Anotamos las columnas que necesiten limpieza y luego estandarizamos los nombres de columnas.

### 2.1 Explorar la estructura y tipos de datos

**Objetivo:**
Identificar columnas con tipos incorrectos, distribución y nulos, y anotar las columnas que requieren conversión.

**Pasos:**

- Utilizamos `.info()` para conocer la estructura de ambos DataFrames.
- Muestramos los primeros 3 renglones de cada DF.
- Identificamos si los detalles de cada DF estan bien o si requieren correcciones y escribimos las conclusiones.


In [ ]:
# Examinar la estructura de traffic
traffic.info()
traffic.head(3)

En la estructura del DF traffic, se observa que:
- Las columnas `UpdateTimeUTC` y `UpdateTimeUTC` son de tipo object, estos datos requieren conversion a tipo fecha.
- Afortunadamente no hay datos ausentes en alguna columna.

In [ ]:
# Examinar la estructura de eco
eco.info()
eco.head(3)

En la estructura del DF eco, se observa que:
- Las columnas `City GDP/capita`, `Unemployment %`se encuentran en formato object, estos datos requiren ser convertidos a formato float64 al ser datos numericos.
- Afortunadamente no hay datos ausentes en alguna columna.

### 2.2 Renombrar columnas

**Objetivo:**
Estandarizar los nombres de columnas para evitar errores y facilitar la unión de los datasets.

**Pasos:**

- Cambiamos los nombres de las columnas para que tengan el formato `snake_case`.
    - `Country` → `country`
    - `UpdateTimeUTC` → `update_time_utc`
- Verificamos que los cambios se hayan aplicado correctamente usando `.columns`.


In [ ]:

# Estandarizar los nombres de las columnas de traffic
traffic = traffic.rename(columns={
    'Country': 'country',
    'City': 'city',
    'UpdateTimeUTC': 'update_time_utc',
    'JamsDelay': 'jams_delay',
    'TrafficIndexLive': 'traffic_index_live',
    'JamsLengthInKms': 'jams_length_kms',
    'JamsCount': 'jams_count',
    'TrafficIndexWeekAgo': 'traffic_index_week_ago',
    'UpdateTimeUTCWeekAgo': 'update_time_utc_week_ago',
    'TravelTimeLivePer10KmsMins': 'travel_time_live_per_10kms_mins',
    'TravelTimeHistoricPer10KmsMins': 'travel_time_hist_per_10kms_mins',
    'MinsDelay': 'mins_delay'
})


# verificar cambios
traffic.columns



In [ ]:
# Estandarizar los nombres de las columnas de eco
eco = eco.rename(columns={'Year':'year','City':'city','Country':'country','City GDP/capita':'city_gdp_capita','Unemployment %':'unemployment','PM2.5 (μg/m³)':'pm2_5_μgm³','Population (M)':'population_m'})
# verificar cambios
eco.columns


### 2.3 Corregir formatos numéricos y de fecha

**Objetivo:**
Asegurar que las columnas de fechas y valores numéricos estén en formatos correctos para permitir análisis, cálculos y comparaciones precisas.

**Pasos:**

- Convertir las columnas de fecha de `traffic` a formato `datetime`. Hacemos el cambio a prueba de errores.
- En el dataset `eco`, limpiamos los valores numéricos:
    - En `city_gdp_capita`: eliminamos separadores de miles (`.`) y reemplazamos las comas (`','`) por puntos (`'.'`) antes de convertirlos a tipo `float`.
    - En `unemployment_pct`: eliminamos el símbolo de porcentaje (`%`) y reemplazamos las comas (`','`) por puntos (`'.'`) antes de convertirlos a tipo `float`.
    - En `population_m`: reemplazamos las comas (`','`) por puntos (`'.'`) antes de convertirlos a tipo `float`.
- Finalmente, creamos una nueva columna llamada `population` multiplicando `population_m` por 1,000,000 para obtener la población total.


In [ ]:
# Convertir las columnas de traffic a tipo fecha con pd.to_datetime()
traffic['update_time_utc'] = pd.to_datetime(traffic['update_time_utc'], errors='coerce', utc=True)
traffic['update_time_utc_week_ago'] = pd.to_datetime(traffic['update_time_utc_week_ago'], errors='coerce', utc=True)

# verificar el cambio
traffic.info()

In [ ]:
# Limpiar separadores y conviertir columnas numéricas en eco
eco['city_gdp_capita'] = (eco['city_gdp_capita'].astype(str).str.replace('.', '').str.replace(',', '.').astype(float))
eco['unemployment'] = (eco['unemployment'].astype(str).str.replace('%', '').str.replace(',', '.').astype(float))
eco['population_m'] = (eco['population_m'].astype(str).str.replace(',', '.').astype(float))

# Calcular la población total en unidades absolutas (Multiplica * 1000000)
eco['population'] = (eco['population_m']* 1000000)

# verificar el cambio
eco.info()
eco.head(3)


---

## Extraer año y filtrar

Extraer el año que permita filtrar la información y trabajar solo con el período más reciente y relevante.

### 3.1 Extraer columna año y filtrar 2024

**Objetivo**
Identificar el año de cada registro y mantener solo los registros del 2024.

**Pasos**

- Como el DataFrame `traffic` no tiene una columna de año, utilizamos el atributo `.dt.year` sobre su columna de fecha para crear una nueva columna llamada `year`.
- Filtramos las filas donde el año sea **2024**.
- Utilizamos `.copy()` para crear dos nuevos DataFrames (`traffic_2024` y `eco_2024`) para evitar modificar el dataset original.

In [ ]:
# Extraer el año de las fechas en update_time_utc
traffic['year'] = traffic['update_time_utc'].dt.year

# Verificar cambio
traffic.head(3)

In [ ]:
# Filtrar los registros del año 2024
traffic_2024 = traffic[traffic['year']==2024].copy()
eco_2024 = eco[eco['year']==2024].copy()

# Revisar dataframes nuevos
display(traffic_2024.head())
display(eco_2024.head())



---

## Analizar y resumir datos de movilidad

Como el dataset de tráfico contiene **múltiples registros por ciudad**. En esta parte, calculamos los promedios anuales por ciudad para simplificar el análisis y obtener una visión más clara de las tendencias generales.

### 4.1 Calcular promedios de tráfico por ciudad

**Objetivo:**
Obtener una vista consolidada del tráfico promedio por ciudad y año, para analizar patrones generales sin depender de datos diarios.

**Pasos**

- Agrupamos los datos por `city`, `country` y `year`.
- Calculamos el promedio **solo de las métricas de tráfico más relevantes**: como `jams_delay`, `traffic_index_live`, `jams_length_kms`, `jams_count`, `mins_delay`, y tiempos de viaje (`travel_time_live_per_10kms_mins` y `travel_time_hist_per_10kms_mins`).
- Guardamos el resultado como `traffic_city_year_2024`, manteniendo las columnas como variables (no índices).


In [ ]:
# Calcular los  promedios de trafico por ciudad, país y año
traffic_city_year_2024 = traffic.groupby(['city','country','year'])['jams_delay', 'traffic_index_live', 'jams_length_kms', 'jams_count', 'mins_delay', 'travel_time_live_per_10kms_mins', 'travel_time_hist_per_10kms_mins'].mean().reset_index()
# Mostrar resultado
traffic_city_year_2024.head()

### **Ciudades con mayor promedio de tráfico**

Para poder conocer las ciudades con mayores promedios de tráfico ejecutamos la siguiente línea de código:

`traffic_city_year_2024.sort_values(["jams_delay"], ascending=False)`



In [ ]:
traffic_city_year_2024.sort_values(["jams_delay"], ascending=False)

La ciudad con el mayor tiempo promedio de tráfico es Mexico City


---

## Unir movilidad y economía

Combinamos datasets que permitan analizar cómo se relacionan los indicadores económicos con los de movilidad.

### 5.1 Unir tráfico (tabla principal) con indicadores económicos

**Objetivo:**
Combinar la información de tráfico y economía en un solo DataFrame para analizar cómo las condiciones económicas se relacionan con la movilidad urbana.

**Pasos**
- Seleccionar solo las **columnas relevantes** de cada dataset (por ejemplo, variables clave de tráfico y de economía).
- Usamos `.copy()` al crear subconjuntos para evitar modificar el dataset original.
- Unimos ambos DataFrames y definimos como **claves de unión** a `city` y `year`.
- Mantemos solo las ciudades y años presentes en ambos datasets.
- Guardamos el resultado en una nueva variable llamada `merged` y muestra las primeras 5 filas.


<details>
<summary>Haz clic para ver la pista</summary>
Aplica una unión de tipo "inner" para mantener las ciudades y años presentes en ambos datasets.

In [ ]:
# Seleccionar columnas clave de tráfico y economía

left_cols = ['city','country','year','jams_delay','traffic_index_live',
             'jams_length_kms','jams_count','mins_delay',
             'travel_time_live_per_10kms_mins','travel_time_hist_per_10kms_mins']

right_cols = ['city','year','city_gdp_capita','unemployment','pm2_5_μgm³','population']


# Usar .copy() para crear los dos nuevos datasets reducidos
traffic_2024_small = traffic_city_year_2024[left_cols].copy()
eco_2024_small = eco_2024[right_cols].copy()
# Unir datasets
merged = pd.merge(traffic_2024_small, eco_2024_small, on=['city','year'], how='inner')

# Mostrar las primeras 5 filas
merged.head()


---

## Visualización y análisis de relaciones

**Visualización de patrones**.
Los gráficos nos ayudarán a entender cómo se relacionan las variables económicas con las de movilidad urbana.

### 6.1 Visualizar relaciones entre economía y tráfico

**Objetivo:**
Analizar visualmente la distribución y la relación entre indicadores de tráfico y economía en 2024, para identificar posibles patrones o tendencias generales entre ambas variables.

**Pasos**
- Usamos las librerías `seaborn` y `matplotlib.pyplot` para generar los gráficos.
- Visualizamos la distribución del **tráfico** (`jams_delay`) mediante:
    - **Boxplot** → para observar la media, mediana y detectar valores atípicos.
- Visualizamos la distribución de la **economía** (`city_gdp_capita`) mediante:
    - **Histograma** → para analizar la forma de la distribución y el valor promedio del PIB per cápita.
- Finalmente, **comparamos ambas variables**, para observar si existe alguna relación entre ellas, haciendo un solo gráfico de barras donde aparezcan ambos indicadores.

In [ ]:
# Crear boxplot para observar el comportamiento de los minutos de congestion JamsDelay
sns.boxplot(data=merged, y='jams_delay', showmeans=True)

# obtener promedio para mostrarlo en título
mean_value = merged['jams_delay'].mean()
plt.title(f'Boxplot de JamsDelay (2024)\nPromedio: {mean_value:.2f}')
plt.show()


In [ ]:
# Crear histograma para ver la distribución de la economía (city_gdp_capita)
merged['city_gdp_capita'].hist(bins=5, figsize=(10,5))
plt.title('Distribución distribución de la economía')
plt.ylabel('city_gdp_capita')
plt.show()


In [ ]:
# Gráfico de barras para comparar jams_delay y city_gdp_capita por ciudad
merged.plot(x='city', y=['jams_delay', 'city_gdp_capita'], kind='bar', figsize=(10,5))
plt.xticks(rotation=90)
plt.show()

### **Analisis**

El análisis muestra que no existe una relación completamente clara entre el PIB per cápita y los niveles de congestión en las ciudades. Aunque en algunos casos las ciudades con mayor desarrollo económico presentan más tráfico, también existen excepciones donde altos ingresos no significan mayor congestión. Esto indica que otros factores, como la infraestructura, el transporte público y la planificación urbana, influyen considerablemente en la movilidad. En conclusión, ambos parámetros solo presentan una ligera relación en ciertos casos, pero no una correlación generalizada.


---

## Exportar y documentar resultados

Para finalizar consolidamos el trabajo: guardamos el dataset limpio y creamos un resumen que documente los resultados del proyecto.

### 7.1 Guardar dataset final

**Objetivo:**
Generamos un CSV limpio, reproducible y con columnas relevantes para análisis posterior.

**Pasos**

- Exportamos el DataFrame `merged` con el nombre: `ladb_mobility_economy_2024_clean.csv`
- Usamos `index=False` para no incluir el índice.


In [ ]:
# Exporta el dataset final como CSV
merged.to_csv("ladb_mobility_economy_2024_clean.csv", index=False)